In [2]:
# Question 1 - Build Personalized Knowledge Base

import pandas as pd

rollNumber = "1024170128"

fixedEntries = [
    {
        "question": "what is the annual fee",
        "answer": "The annual fee is Rs 500.",
        "keywords": "fee cost price charge",
        "category": "billing",
    },
    {
        "question": "how to reset password",
        "answer": "Go to Settings > Reset Password.",
        "keywords": "password reset login",
        "category": "account",
    },
    {
        "question": "what are your working hours",
        "answer": "We are open 9 AM to 5 PM.",
        "keywords": "hours timing open time",
        "category": "general",
    },
    {
        "question": "how can i pay the fee",
        "answer": "You can pay via UPI, card, or net banking.",
        "keywords": "pay payment upi fee",
        "category": "billing",
    },
]

personalEntries = [
    {
        "question": "how do i update my registered mobile number",
        "answer": "Go to Profile > Account Settings to update it.",
        "keywords": "mobile phone update",
        "category": "account",
    },
    {
        "question": "can i get a fee payment receipt",
        "answer": "Yes, download it from the Payments section.",
        "keywords": "receipt invoice payment",
        "category": "billing",
    },
]

df = pd.DataFrame(fixedEntries + personalEntries)

print("Final 6-row Knowledge Base")
df

Final 6-row Knowledge Base


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how do i update my registered mobile number,Go to Profile > Account Settings to update it.,mobile phone update,account
5,can i get a fee payment receipt,"Yes, download it from the Payments section.",receipt invoice payment,billing


In [3]:
# Question 2 - Generate and Score a Hypothesis

def score_query(query, df):
    queryWords = query.lower().split()
    results = []

    for _, row in df.iterrows():
        text = (
            row["question"] + " " +
            row["keywords"] + " " +
            row["category"]
        ).lower()

        score = sum(word in text for word in queryWords)

        if score > 0:
            confidence = score / len(queryWords)
            results.append({
                "Question": row["question"],
                "Category": row["category"],
                "Confidence": confidence
            })

    return pd.DataFrame(results).sort_values(
        by="Confidence",
        ascending=False
    )

score_query("fee payment", df)

,Question,Category,Confidence
1,how can i pay the fee,billing,1.0
2,can i get a fee payment receipt,billing,1.0
0,what is the annual fee,billing,0.5


In [4]:
# Question 3 - Same Category Function

def sameCategory(categoryName, df):
    return df[df["category"] == categoryName][["question"]]

# Personalized category = account
sameCategory("account", df)

,question
1,how to reset password
4,how do i update my registered mobile number


In [5]:
# Question 4 - Update Keywords and Save CSV

entry_index = 4   # Personalized account entry

newKeyword = input("Enter a new keyword: ")

df.at[entry_index, "keywords"] += " " + newKeyword

fileName = "67_faq_data.csv"
df.to_csv(fileName, index=False)

print("CSV saved successfully as:", fileName)

df

CSV saved successfully as: 67_faq_data.csv


,question,answer,keywords,category
0,what is the annual fee,The annual fee is Rs 500.,fee cost price charge,billing
1,how to reset password,Go to Settings > Reset Password.,password reset login,account
2,what are your working hours,We are open 9 AM to 5 PM.,hours timing open time,general
3,how can i pay the fee,"You can pay via UPI, card, or net banking.",pay payment upi fee,billing
4,how do i update my registered mobile number,Go to Profile > Account Settings to update it.,mobile phone update hi,account
5,can i get a fee payment receipt,"Yes, download it from the Payments section.",receipt invoice payment,billing


In [6]:
# Question 5 - Group By Category

category_count = df.groupby("category").size().reset_index(name="Count")

category_count

,category,Count
0,account,2
1,billing,3
2,general,1


In [7]:
# Question 6 - Tie Handling in Scoring Function

def best_match(query, df):
    query_words = query.lower().split()
    scores = []

    for _, row in df.iterrows():
        text = (
            row["question"] + " " +
            row["keywords"] + " " +
            row["category"]
        ).lower()

        score = sum(word in text for word in query_words)
        scores.append(score)

    temp = df.copy()
    temp["score"] = scores

    highest = temp["score"].max()

    if highest == 0:
        print("No match found.")
        return

    top = temp[temp["score"] == highest]

    if len(top) > 1:
        print("Tie Found")
    else:
        print("Best Match")

    return top[["question", "category", "score"]]

print("Tie Query: fee")
display(best_match("fee", df))

print("\nNon-Tie Query: password")
display(best_match("password", df))

Tie Query: fee
Tie Found


,question,category,score
0,what is the annual fee,billing,1
3,how can i pay the fee,billing,1
5,can i get a fee payment receipt,billing,1



Non-Tie Query: password
Best Match


,question,category,score
1,how to reset password,account,1
